# Bhagavad Gita -> chanted audio (Vāgdhenu TTS)

Runtime -> Change runtime type -> **T4 GPU**. Run cells top to bottom. Output goes to Google Drive (`MyDrive/gita_audio`).
Resumable: if Colab disconnects, reconnect and re-run all cells; finished verses are skipped.

Colab now ships Python 3.13, but Vāgdhenu needs Python 3.10 with pinned packages. So everything model-related runs in a separate **Python 3.10 virtualenv** (`/content/venv`) made with `uv`; it never touches Colab's own packages.

Model: https://huggingface.co/prathoshap/vagdhenu (Apache-2.0; vocoder is a BigVGAN-v2 derivative, observe NVIDIA's terms; do not impersonate the voice).
Text: https://github.com/gita/gita (check its license before redistributing the audio).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L

In [ ]:
%%bash
# One-time setup (~5-10 min). Mirrors scripts/setup.sh, but inside a Python 3.10 venv.
set -e
pip install -q uv
cd /content
[ -d vagdhenu ] || git clone --depth 1 https://github.com/prathoshap/vagdhenu.git
cd vagdhenu
[ -d /content/venv ] || uv venv --python 3.10 /content/venv
PY=/content/venv/bin/python
uv pip install -q --python $PY torch==2.4.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
uv pip install -q --python $PY -r requirements.txt
# requirements.txt can pull a newer torch/torchaudio (torchaudio>=2.9 needs torchcodec): re-pin the validated versions
uv pip install -q --python $PY torch==2.4.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://pypi.org/simple --index-strategy unsafe-best-match
$PY -c "import torch, torchaudio, numpy; print('torch', torch.__version__, 'torchaudio', torchaudio.__version__, 'numpy', numpy.__version__)"
[ -d BigVGAN/.git ] || git clone --depth 1 https://github.com/NVIDIA/BigVGAN.git BigVGAN
PYTHONPATH="$PWD/BigVGAN" $PY scripts/download_weights.py
ls models
echo setup done

In [ ]:
# Build per-verse render jobs from the Gita text (Devanagari)
import json, os, re, sys, subprocess, urllib.request
os.chdir('/content/vagdhenu')
PY = '/content/venv/bin/python'
ENV = dict(os.environ, PYTHONPATH=os.getcwd() + '/BigVGAN:' + os.getcwd() + '/src', MPLBACKEND='Agg')
OUT = '/content/drive/MyDrive/gita_audio'
os.makedirs(OUT + '/verses', exist_ok=True)

url = 'https://raw.githubusercontent.com/gita/gita/main/data/verse.json'
verses = json.load(urllib.request.urlopen(url))
print(len(verses), 'verse records')

SPK = re.compile(r'^(धृतराष्ट्र उवाच|(?:सञ्जय|संजय) उवाच|अर्जुन उवाच|श्री ?भगवानुवाच)\s*')   # speaker (uvach) prefixes

def to_lines(text):
    lines = []
    for ln in text.split('\n'):
        ln = ln.strip()
        if not ln:
            continue
        ln = re.sub(r'।।\s*[\d.\-]+\s*।।', '', ln)   # verse-number marker
        ln = SPK.sub('', ln).strip()                  # drop a standalone speaker line, or the speaker glued to the verse
        ln = ln.rstrip('।॥').strip()
        if ln:
            lines.append(ln)
    return lines

shards = []
for v in verses:
    lines = to_lines(v['text'])
    if not lines:
        continue
    padas = [l + (' ॥' if i == len(lines) - 1 else ' ।') for i, l in enumerate(lines)]
    shards.append({'id': f"{v['chapter_number']:02d}_{v['verse_number']:03d}", 'meter': 'anuṣṭubh',
                   'padas': padas, 'seed': 60, 'no_sandhi': True})

# Meter detection runs inside the py3.10 venv (render_core needs torch). Falls back to anuṣṭubh on any error.
open('/content/detect.py', 'w').write('''
import json, sys
shards = json.load(open('/content/shards.json', encoding='utf8'))
try:
    from render_core import detect_meter_key
except Exception as e:
    print('detector unavailable:', e); sys.exit(0)
for s in shards:
    try:
        m = detect_meter_key(' '.join(s['padas']))
        if m: s['meter'] = m
    except Exception:
        pass
json.dump(shards, open('/content/shards.json', 'w', encoding='utf8'), ensure_ascii=False)
''')
json.dump(shards, open('/content/shards.json', 'w', encoding='utf8'), ensure_ascii=False)
r = subprocess.run([PY, '/content/detect.py'], env=ENV, capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:])
shards = json.load(open('/content/shards.json', encoding='utf8'))
print(len(shards), 'verses to render')
from collections import Counter; print(Counter(s['meter'] for s in shards))

In [ ]:
# TEST FIRST: render chapter 1, verses 1-3 and listen before committing to the full run
import IPython.display as ipd
def render(jobs, tag):
    jobs = [j for j in jobs if not os.path.exists(f"{OUT}/verses/{j['id']}.wav")]
    if not jobs:
        return
    json.dump(jobs, open(f'/content/{tag}.json', 'w'), ensure_ascii=False)
    r = subprocess.run([PY, 'src/render.py', '--shard', f'/content/{tag}.json',
                        '--results', f'/content/{tag}_res.json', '--outdir', OUT + '/verses'],
                       env=ENV, capture_output=True, text=True)
    print('exit code', r.returncode); print((r.stdout or '')[-1500:]); print((r.stderr or '')[-3000:])
    if r.returncode != 0:
        raise RuntimeError('render.py failed: ' + (r.stderr or '')[-2500:])
render(shards[:3], 'test')
for s in shards[:3]:
    p = f"{OUT}/verses/{s['id']}.wav"
    if os.path.exists(p): ipd.display(ipd.Audio(p))
    else: print('NOT CREATED:', p)

In [ ]:
# FULL RUN: all 18 chapters, resumable (skips existing wavs)
for ch in range(1, 19):
    jobs = [s for s in shards if s['id'].startswith(f'{ch:02d}_')]
    print('chapter', ch, len(jobs), 'verses')
    render(jobs, f'ch{ch:02d}')

In [ ]:
# Stitch verses into one audio file per chapter (+ mp3)
import numpy as np, soundfile as sf
os.makedirs(OUT + '/chapters', exist_ok=True)
SR, GAP = 24000, np.zeros(int(24000 * 1.2), dtype='float32')
missing = []
for ch in range(1, 19):
    parts = []
    for s in [s for s in shards if s['id'].startswith(f'{ch:02d}_')]:
        p = f"{OUT}/verses/{s['id']}.wav"
        if not os.path.exists(p):
            missing.append(s['id']); continue
        a, sr = sf.read(p, dtype='float32'); assert sr == SR, sr
        if a.ndim > 1: a = a.mean(1)
        parts += [a, GAP]
    if parts:
        wav = f'{OUT}/chapters/gita_ch{ch:02d}.wav'
        sf.write(wav, np.concatenate(parts), SR)
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', wav, '-b:a', '96k', wav[:-4] + '.mp3'])
print('missing verses (re-run the full-run cell):', missing)

## Timestamps (for syncing text with the audio)
Run these after the stitch cell. They produce, in "MyDrive/gita_audio/timestamps": a JSON per chapter, plus .lrc and .srt files.
1. **Cell A**: verse- and pāda-level times (computed from the audio itself).
2. **Cell B (optional)**: word-level times by forced alignment (ctc-forced-aligner, Meta MMS model, in its own venv). Test mode aligns 3 verses first; set TEST = False for everything. Word times are approximate for stretched chant vowels, so spot-check by ear.
3. **Cell C**: write the JSON / LRC / SRT files (uses word times if Cell B has run).

In [ ]:
# TIMESTAMPS A: verse- and pada-level times, as offsets inside each chapter file (matches the stitch cell)
import numpy as np, soundfile as sf, json, os
SR, GAP_S = 24000, 1.2          # same as the stitch cell
os.makedirs(OUT + '/timestamps', exist_ok=True)

def split_padas(a, n, sr=SR, hop_s=0.01):
    """Return n (start, end) spans in seconds, using the n-1 longest internal silences of a verse wav."""
    hop = int(hop_s * sr); frames = len(a) // hop
    env = np.sqrt((a[:frames * hop].reshape(frames, hop) ** 2).mean(axis=1))
    voiced = env > 0.03 * env.max()
    idx = np.where(voiced)[0]
    first, last = idx[0], idx[-1]
    if n == 1:
        return [(first * hop_s, (last + 1) * hop_s)], True
    runs, i = [], first
    while i <= last:                      # silent runs between first and last voiced frame
        if not voiced[i]:
            j = i
            while j <= last and not voiced[j]: j += 1
            runs.append((j - i, i, j)); i = j
        else:
            i += 1
    if len(runs) < n - 1:                 # fallback: equal split (flagged approximate)
        edges = np.linspace(first, last + 1, n + 1) * hop_s
        return [(edges[k], edges[k + 1]) for k in range(n)], False
    cuts = sorted(sorted(runs, reverse=True)[:n - 1], key=lambda r: r[1])
    starts = [first] + [c[2] for c in cuts]
    ends = [c[1] for c in cuts] + [last + 1]
    return [(s * hop_s, e * hop_s) for s, e in zip(starts, ends)], True

TS, approx = {}, []
for ch in range(1, 19):
    t0, rows = 0.0, []
    for s in [s for s in shards if s['id'].startswith(f'{ch:02d}_')]:
        p = f"{OUT}/verses/{s['id']}.wav"
        if not os.path.exists(p): continue
        a, sr = sf.read(p, dtype='float32')
        if a.ndim > 1: a = a.mean(1)
        dur = len(a) / sr
        spans, ok = split_padas(a, len(s['padas']))
        if not ok: approx.append(s['id'])
        rows.append({'id': s['id'], 'start': round(t0, 3), 'end': round(t0 + dur, 3),
                     'text': ' '.join(s['padas']),
                     'padas': [{'text': t, 'start': round(t0 + x0, 3), 'end': round(t0 + x1, 3)}
                               for t, (x0, x1) in zip(s['padas'], spans)]})
        t0 += dur + GAP_S
    TS[ch] = rows
print({ch: len(r) for ch, r in TS.items()})
print('verses with approximate pada split (equal-split fallback):', approx[:20], '...' if len(approx) > 20 else '')
print(json.dumps(TS[1][0], ensure_ascii=False, indent=1)[:700])

In [ ]:
# TIMESTAMPS B (optional): word-level alignment with ctc-forced-aligner (Meta MMS) in its own venv
# Set TEST = False to align every verse. Resumable: existing results are skipped.
TEST = True
import subprocess, json, os, re
os.makedirs(OUT + '/timestamps/words', exist_ok=True)
r = subprocess.run('''set -e
pip install -q uv
[ -d /content/venv_align ] || uv venv --python 3.11 /content/venv_align
uv pip install -q --python /content/venv_align/bin/python torch "git+https://github.com/MahmoudAshraf97/ctc-forced-aligner.git"
''', shell=True, capture_output=True, text=True, executable='/bin/bash')
print('install exit', r.returncode, r.stderr[-1500:])

def plain(t): return re.sub(r'[।॥]', ' ', t).split()
vs = [s for s in shards if os.path.exists(f"{OUT}/verses/{s['id']}.wav")]
if TEST: vs = vs[:3]
jobs = [{'id': s['id'], 'wav': f"{OUT}/verses/{s['id']}.wav", 'text': ' '.join(plain(' '.join(s['padas']))),
         'out': f"{OUT}/timestamps/words/{s['id']}.json"} for s in vs]
json.dump(jobs, open('/content/align_jobs.json', 'w'), ensure_ascii=False)

open('/content/align.py', 'w').write('''
import json, os, torch
from ctc_forced_aligner import (load_audio, load_alignment_model, generate_emissions, preprocess_text,
                                get_alignments, get_spans, postprocess_results)
jobs = json.load(open('/content/align_jobs.json', encoding='utf8'))
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model, tok = load_alignment_model(dev, dtype=torch.float16 if dev == 'cuda' else torch.float32)
for j in jobs:
    if os.path.exists(j['out']): continue
    try:
        audio = load_audio(j['wav'], model.dtype, model.device)
        em, stride = generate_emissions(model, audio, batch_size=4)
        tokens = None
        for lang in ('san', 'hin'):          # Sanskrit first, Hindi romanisation as fallback
            try:
                tokens, text = preprocess_text(j['text'], romanize=True, language=lang); break
            except Exception as e:
                err = e
        if tokens is None: raise err
        seg, scores, blank = get_alignments(em, tokens, tok)
        words = postprocess_results(text, get_spans(tokens, seg, blank), stride, scores)
        words = [w for w in words if w['text'] != '<star>']
        json.dump([{'text': w['text'], 'start': round(w['start'], 3), 'end': round(w['end'], 3)} for w in words],
                  open(j['out'], 'w', encoding='utf8'), ensure_ascii=False)
    except Exception as e:
        print('ALIGN FAIL', j['id'], repr(e))
print('align done')
''')
r = subprocess.run(['/content/venv_align/bin/python', '/content/align.py'], capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-1500:] if r.returncode else '')
for j in jobs[:3]:
    if os.path.exists(j['out']): print(j['id'], open(j['out'], encoding='utf8').read()[:400])

In [ ]:
# TIMESTAMPS C: write JSON / LRC / SRT per chapter (word times are included where Cell B produced them)
def lrc_t(x): return f"{int(x // 60):02d}:{x % 60:05.2f}"
def srt_t(x): return f"{int(x // 3600):02d}:{int(x % 3600 // 60):02d}:{int(x % 60):02d},{int(round((x % 1) * 1000)):03d}"
nwords = 0
for ch, rows in TS.items():
    if not rows: continue
    lrc, srt, n = [], [], 1
    for r in rows:
        wp = f"{OUT}/timestamps/words/{r['id']}.json"
        if os.path.exists(wp):
            r['words'] = [dict(text=w['text'], start=round(r['start'] + w['start'], 3), end=round(r['start'] + w['end'], 3))
                          for w in json.load(open(wp, encoding='utf8'))]
            nwords += len(r['words'])
        for p in r['padas']:
            lrc.append(f"[{lrc_t(p['start'])}]{p['text']}")
            srt += [str(n), f"{srt_t(p['start'])} --> {srt_t(p['end'])}", p['text'], '']; n += 1
    base = f"{OUT}/timestamps/gita_ch{ch:02d}"
    json.dump(rows, open(base + '.json', 'w', encoding='utf8'), ensure_ascii=False, indent=1)
    open(base + '.lrc', 'w', encoding='utf8').write('\n'.join(lrc) + '\n')
    open(base + '.srt', 'w', encoding='utf8').write('\n'.join(srt))
print('written to', OUT + '/timestamps', '| word entries included:', nwords)